- We are given a table of Uber rides that contains information about the mileage and the purpose of the business expense. Our task is to find the top 3 business purpose categories that generate the most miles driven by passengers using Uber for business transportation.

#### Approach
- First, we filter the records to include only those where the category is 'Business'.
- We calculate the total miles for each purpose using group by.
- We sort the data by total_miles in descending order and select top 3 using the limit(3) clause.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as sum_
from pyspark.sql.types import StructType, StructField, StringType, FloatType


In [0]:
spark = SparkSession.builder.appName("TopBusinessPurposes").getOrCreate()


In [0]:
data = [
    ('2016-01-01 21:11', '2016-01-01 21:17', 'Business', 'Fort Pierce', 'Fort Pierce', 5.1, 'Meal/Entertain'),
    ('2016-01-02 01:25', '2016-01-02 01:37', 'Business', 'Fort Pierce', 'Fort Pierce', 5, None),
    ('2016-01-02 20:25', '2016-01-02 20:38', 'Business', 'Fort Pierce', 'Fort Pierce', 4.8, 'Errand/Supplies'),
    ('2016-01-05 17:31', '2016-01-05 17:45', 'Business', 'Fort Pierce', 'Fort Pierce', 4.7, 'Meeting'),
    ('2016-01-06 14:42', '2016-01-06 15:49', 'Business', 'Fort Pierce', 'West Palm Beach', 63.7, 'Customer Visit'),
    ('2016-01-06 17:15', '2016-01-06 17:19', 'Business', 'West Palm Beach', 'West Palm Beach', 4.3, 'Meal/Entertain'),
    ('2016-01-06 17:30', '2016-01-06 17:35', 'Business', 'West Palm Beach', 'Palm Beach', 7.1, 'Meeting')
]

In [0]:
schema = StructType([
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("category", StringType(), True),
    StructField("start", StringType(), True),
    StructField("stop", StringType(), True),
    StructField("miles", FloatType(), True),
    StructField("purpose", StringType(), True)
])

In [0]:
data_with_float_miles = [(start_date, end_date, category, start, stop, float(miles), purpose)
                         for (start_date, end_date, category, start, stop, miles, purpose) in data]


In [0]:
df = spark.createDataFrame(data_with_float_miles, schema)

In [0]:
df_business = df.filter(col('category') == 'Business')


In [0]:
result_df = df_business.groupBy('purpose').agg(sum_('miles').alias('total_miles'))


In [0]:
result_df = result_df.orderBy(col('total_miles').desc()).limit(3)


In [0]:
result_df.show()


+--------------+------------------+
|       purpose|       total_miles|
+--------------+------------------+
|Customer Visit| 63.70000076293945|
|       Meeting|11.799999713897705|
|Meal/Entertain| 9.400000095367432|
+--------------+------------------+

